# 03 · RAG retrieval and risk analysis walkthrough

Reproduces the guide's worked example: a clause with a 7-day termination notice is matched against the legal knowledge base and the project reference template, then assessed for potential risk.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)

In [2]:
from backend.services import rag_service, risk_analyzer, summarizer
clause = {'text': 'Either party can terminate with 7 days notice.', 'category': 'Termination', 'title': 'Termination'}
refs = rag_service.retrieve_knowledge(clause['text'], 'employment', 'Termination', k=3)
for r in refs: print(round(r['score'],3), r['metadata']['source'], '|', r['metadata']['citation'][:90])

Sentence Transformers unavailable: No module named 'sentence_transformers' — using LSA fallback


0.243 employment/termination.txt | Industrial Disputes Act, 1947, sections 2(s) and 25F; Industrial Relations Code, 2020; Cen
0.179 employment/termination.txt | Industrial Disputes Act, 1947, sections 2(s) and 25F; Industrial Relations Code, 2020; Cen
0.163 employment/termination.txt | Industrial Disputes Act, 1947, sections 2(s) and 25F; Industrial Relations Code, 2020; Cen


In [3]:
std = rag_service.retrieve_standard(clause['text'], 'employment', 'Termination', k=1)[0]
print(std['metadata']['citation']); print(std['text'])

Project reference template (not an official legal standard)
Either party may terminate this Agreement by giving thirty (30) days' written notice to the other, or salary in lieu of the notice period. The Employer may terminate immediately for proven misconduct after giving the Employee a fair opportunity to be heard. All earned salary and dues shall be paid within the time required by law after termination.


In [4]:
summary, method = summarizer.explain_clause(clause, refs)
print(method); print(summary)

rule-based plain-language rewrite
In simple terms: Either party can end with 7 days notice.
Key details — time periods mentioned: 7 days.


In [5]:
r = risk_analyzer.analyze(clause, 'employment', std, refs, {})
print(r['risk_level'], '-', r['method']); print(r['risk_reason']); print('Precaution:', r['precaution'])

HIGH - rule-based risk signals
Notice period of 7 days (Either party): The notice period is shorter than the 30 days used in the reference clause, leaving little time to respond or find alternatives.
Precaution: Ask for at least 30 days' written notice, applying equally to both parties.
